# Training News Classification Models

This notebook demonstrates:
1. Training a baseline model (TF-IDF + Logistic Regression)
2. Training a deep learning model (TensorFlow/Keras)
3. Comparing model performance
4. Making predictions

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Load and Prepare Data

In [ ]:
from data_loading.news_dataset import NewsDataset

# Load dataset
dataset = NewsDataset(csv_path='../data/news.csv')
dataset.load_data()
dataset.clean_data()

# Split data
train_df, val_df, test_df = dataset.prepare_splits(
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

# Extract texts and labels
train_texts, train_labels = dataset.get_texts_and_labels(train_df)
val_texts, val_labels = dataset.get_texts_and_labels(val_df)
test_texts, test_labels = dataset.get_texts_and_labels(test_df)

class_names = ['neutral', 'left', 'right', 'propaganda']

## 2. Baseline Model: TF-IDF + Logistic Regression

In [ ]:
from preprocessing.text_preprocessing import TextPreprocessor, TfidfFeatureExtractor
from models.news_baseline_sklearn import NewsBaselineModel

# Preprocess texts
print("Preprocessing texts...")
preprocessor = TextPreprocessor(remove_stopwords=True, lowercase=True)

train_texts_clean = preprocessor.clean_texts(train_texts)
val_texts_clean = preprocessor.clean_texts(val_texts)
test_texts_clean = preprocessor.clean_texts(test_texts)

print("Sample cleaned text:")
print(train_texts_clean[0][:200] + "...")

In [ ]:
# Extract TF-IDF features
print("Extracting TF-IDF features...")
tfidf = TfidfFeatureExtractor(max_features=5000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(train_texts_clean)
X_val = tfidf.transform(val_texts_clean)
X_test = tfidf.transform(test_texts_clean)

print(f"Feature matrix shape: {X_train.shape}")

In [ ]:
# Train baseline model
print("Training baseline model...")
baseline_model = NewsBaselineModel(model_type='logistic', max_iter=1000)
baseline_model.train(X_train, train_labels, X_val, val_labels)

In [ ]:
# Evaluate baseline model
baseline_results = baseline_model.evaluate(X_test, test_labels, class_names)

# Confusion matrix
cm = confusion_matrix(test_labels, baseline_results['predictions'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Baseline Model - Confusion Matrix', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 3. Deep Learning Model: TensorFlow/Keras

In [ ]:
from models.news_deep_tf import NewsDeepModel
import tensorflow as tf

# Check GPU
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Light preprocessing for deep learning (keep more words)
preprocessor_dl = TextPreprocessor(remove_stopwords=False, lowercase=True)

train_texts_dl = preprocessor_dl.clean_texts(train_texts)
val_texts_dl = preprocessor_dl.clean_texts(val_texts)
test_texts_dl = preprocessor_dl.clean_texts(test_texts)

In [ ]:
# Build deep learning model
deep_model = NewsDeepModel(
    vocab_size=10000,
    embedding_dim=128,
    max_length=200,
    num_classes=4,
    architecture='cnn',  # or 'lstm'
    random_state=42
)

deep_model.build_model()

In [ ]:
# Train deep learning model
print("Training deep learning model...")
history = deep_model.train(
    train_texts=train_texts_dl,
    train_labels=train_labels,
    val_texts=val_texts_dl,
    val_labels=val_labels,
    epochs=15,
    batch_size=32
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train', marker='o')
ax1.plot(history.history['val_accuracy'], label='Validation', marker='o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy', fontweight='bold')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], label='Train', marker='o')
ax2.plot(history.history['val_loss'], label='Validation', marker='o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss', fontweight='bold')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate deep learning model
deep_preds = deep_model.predict(test_texts_dl)

from sklearn.metrics import accuracy_score
deep_accuracy = accuracy_score(test_labels, deep_preds)

print(f"Deep Learning Test Accuracy: {deep_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, deep_preds, target_names=class_names))

# Confusion matrix
cm_deep = confusion_matrix(test_labels, deep_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_deep, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Deep Learning Model - Confusion Matrix', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 4. Model Comparison

In [ ]:
# Compare accuracies
comparison = {
    'Model': ['Baseline (TF-IDF + Logistic)', 'Deep Learning (CNN/LSTM)'],
    'Accuracy': [baseline_results['accuracy'], deep_accuracy]
}

import pandas as pd
comparison_df = pd.DataFrame(comparison)
print(comparison_df)

# Visualize comparison
plt.figure(figsize=(10, 6))
plt.bar(comparison_df['Model'], comparison_df['Accuracy'], 
        color=['skyblue', 'lightgreen'], edgecolor='black')
plt.ylabel('Accuracy')
plt.title('Model Comparison', fontweight='bold', fontsize=14)
plt.ylim(0, 1)
plt.xticks(rotation=15, ha='right')

# Add value labels on bars
for i, v in enumerate(comparison_df['Accuracy']):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Make Predictions on New Text

In [ ]:
# Example prediction
sample_text = """Breaking news: The government announced major policy reforms today.
Officials stated that these changes will improve the system for everyone.
The new regulations will take effect next month."""

print("Sample text:")
print(sample_text)
print("\n" + "="*70)

# Baseline prediction
sample_clean = preprocessor.clean_text(sample_text)
sample_tfidf = tfidf.transform([sample_clean])
baseline_pred = baseline_model.predict(sample_tfidf)[0]
baseline_probs = baseline_model.predict_proba(sample_tfidf)[0]

print("\nBaseline Model Prediction:")
print(f"Class: {class_names[baseline_pred]}")
print(f"Confidence: {baseline_probs[baseline_pred]*100:.2f}%")

# Deep learning prediction
sample_dl = preprocessor_dl.clean_text(sample_text)
deep_pred = deep_model.predict([sample_dl])[0]
deep_probs = deep_model.predict_proba([sample_dl])[0]

print("\nDeep Learning Model Prediction:")
print(f"Class: {class_names[deep_pred]}")
print(f"Confidence: {deep_probs[deep_pred]*100:.2f}%")

# Show all probabilities
print("\n" + "="*70)
print("All Class Probabilities (Deep Learning):")
for i, class_name in enumerate(class_names):
    print(f"  {class_name:12s}: {deep_probs[i]*100:6.2f}%")

## 6. Conclusions

Key takeaways:

1. **Baseline Model**: Fast to train, interpretable, good starting point
2. **Deep Learning Model**: Can capture complex patterns, may perform better with more data
3. **Trade-offs**: Deep learning requires more computation but can achieve higher accuracy
4. **Practical Use**: Choose model based on your constraints (time, compute, accuracy needs)

Next steps:
- Try different architectures (LSTM vs CNN)
- Experiment with hyperparameters
- Use pre-trained embeddings (word2vec, GloVe)
- Collect more training data